# Comparative genomics with BLAST
Opened from **BNERC**, the Bionitrogen Fixation Research Database (BicBioEng lab, University of South Dakota).

Downloads each selected genome, builds one BLAST database from all of them, then searches it with the first genome as the query and plots the top hits.

**How to run:** paste the records BNERC copied into `RECORDS` below, then choose **Runtime → Run all**.

In [ ]:
#@title Records to analyze
#@markdown Paste what BNERC copied for you: `accession:source` pairs separated by commas. The source can be left out; it's inferred from the accession.
RECORDS = "GCA_000009705:cyanobacteria_ena, GCA_000020025:cyanobacteria_ena"  #@param {type:"string"}
#@markdown NCBI asks for a contact email with each download request.
NCBI_EMAIL = "readstech@gmail.com"  #@param {type:"string"}

def _parse(text):
    out = []
    for part in text.replace("\n", ",").split(","):
        part = part.strip()
        if not part:
            continue
        acc, _, src = part.partition(":")
        acc, src = acc.strip(), src.strip()
        if not src:
            src = "cyanobacteria_ncbi" if acc.upper().startswith(("GCA_", "GCF_")) else "cyanorak"
        out.append((acc, src))
    return out

records = _parse(RECORDS)
accessions = [a for a, _ in records]
assert records, "Paste at least one record into RECORDS above."
print(f"{len(records)} record(s):", ", ".join(f"{a} ({s})" for a, s in records))

## 1. Install tools

In [ ]:
!pip -q install biopython pandas matplotlib
!apt-get -qq install -y ncbi-blast+ > /dev/null && blastn -version | head -1

## 2. Download sequences

In [ ]:
#@title Download genomes as FASTA (.fna)
import glob, os, shutil, subprocess, urllib.request, zipfile
from Bio import Entrez
Entrez.email = NCBI_EMAIL

def _datasets_cli():
    exe = shutil.which("datasets") or os.path.abspath("bin/datasets")
    if not os.path.exists(exe):
        os.makedirs("bin", exist_ok=True)
        urllib.request.urlretrieve("https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets", exe)
        os.chmod(exe, 0o755)
    return exe

def fetch(acc, source):
    dest = f"{acc}.fna"
    if os.path.exists(dest):
        return print(f"✅ {dest} already downloaded")
    try:
        if source in ("cyanobacteria_ncbi", "cyanobacteria_gtdb"):
            # Assembly accessions (GCA_/GCF_): NCBI Datasets CLI
            subprocess.run([_datasets_cli(), "download", "genome", "accession", acc, "--include", "genome", "--filename", f"{acc}.zip"], check=True)
            with zipfile.ZipFile(f"{acc}.zip") as z:
                z.extractall(f"genome_{acc}")
            hits = glob.glob(f"genome_{acc}/ncbi_dataset/data/*/*.fna")
            if not hits:
                return print(f"❌ No .fna in the NCBI package for {acc}")
            shutil.move(hits[0], dest)
        elif source == "cyanobacteria_ena":
            urllib.request.urlretrieve(f"https://www.ebi.ac.uk/ena/browser/api/fasta/{acc}?download=true", dest)
        else:
            # cyanorak / cyanobacteria_ncbi_nucelotide: NCBI Nucleotide via Entrez
            with Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text") as h:
                open(dest, "w").write(h.read())
        print(f"✅ {dest} ({os.path.getsize(dest) // 1024} KB)")
    except Exception as e:
        print(f"❌ {acc} ({source}): {e}")

for acc, src in records:
    fetch(acc, src)
missing = [a for a in accessions if not os.path.exists(f"{a}.fna") or os.path.getsize(f"{a}.fna") == 0]
if missing:
    print("⚠️ Not downloaded:", ", ".join(missing))

## 3. Build one BLAST database

In [ ]:
with open("all_sequences.fna", "w") as out:
    for acc in accessions:
        if os.path.exists(f"{acc}.fna"):
            out.write(open(f"{acc}.fna").read())
!makeblastdb -in all_sequences.fna -dbtype nucl -out combined_db

## 4. BLAST the first genome against all of them

In [ ]:
query = accessions[0]
print("Query:", query)
!blastn -query {query}.fna -db combined_db -out blast_results.txt -outfmt "6 qseqid sseqid length pident bitscore evalue" -max_target_seqs 50

## 5. Top hits

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
cols = ["qseqid", "sseqid", "length", "pident", "bitscore", "evalue"]
df = pd.read_csv("blast_results.txt", sep="\t", names=cols).sort_values("bitscore", ascending=False).head(10)
display(df)
plt.figure(figsize=(10, 5))
plt.barh(df["sseqid"], df["bitscore"], color="#0E8C8C")
plt.title(f"Top BLAST hits for {query}")
plt.xlabel("Bit score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()